# Explainability of a Frozen AI Text Detector

This notebook evaluates the **original pretrained** `dinisds/stylometric-ai-detector` on a balanced 10,000-record RAID sample and explains that detector with InterpretML. It does not train, fine-tune, tune thresholds, or substitute a classifier. A prediction is not proof of authorship.

The first model load uses Joblib/pickle. Review `models/README.md` and the pinned model artifact SHA-256 before setting `ALLOW_PICKLE_MODEL=1`. RAID's full labeled English train split is streamed to create a reproducible reservoir sample; this can take time and transfer substantial data.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import interpret
import datasets
import huggingface_hub
import joblib
import yaml

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    raise RuntimeError('Open this notebook from the project root or notebooks/ directory.')
sys.path.insert(0, str(ROOT))
for folder in ['data', 'outputs/metrics', 'outputs/figures', 'outputs/explanations', 'outputs/reports']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

from src.feature_extraction import FEATURES
from src.model_loader import load_frozen_model
from src.dataset_loader import sample_raid
from src.evaluation import evaluate_model
from src.explainability import compatibility_gate, build_explainers
from src.what_if_analysis import one_feature_what_if

CONFIG = yaml.safe_load((ROOT / 'configs' / 'experiment.yaml').read_text())
SEED = int(CONFIG['dataset']['seed'])


## 1. Load and verify the exact published artifact

The loader fetches the artifact at the pinned Hugging Face commit, checks its SHA-256 before deserialization, and confirms its class labels, feature count, and 100-estimator Random Forest contract. The model is never fit.


In [ ]:
model, model_metadata = load_frozen_model(cache_dir=ROOT / 'models' / 'cache')
model_metadata['download_date_utc'] = datetime.now(timezone.utc).date().isoformat()
model_metadata


## 2. Build the independent balanced RAID evaluation sample

We use the labeled `raid` / `train` split pinned in the configuration. RAID's card describes this as English genres; the `extra` split includes code/Czech/German, and `test` has no labels. Nonempty text with `model == human` is human class 0; named generator models are AI class 1. The reservoir scan keeps available IDs, source IDs, domain, model, attack, decoding, repetition, and title metadata. If either class is short of 5,000 eligible examples, the loader stops instead of changing the target.


In [ ]:
records, sample_metadata = sample_raid(
    per_class=int(CONFIG['dataset']['sample_per_class']),
    seed=SEED, config=CONFIG['dataset']['config'], split=CONFIG['dataset']['split'],
    exclusions_path=ROOT / 'data' / 'excluded_record_ids.csv',
)
manifest_columns = ['id', 'source_id', 'adv_source_id', 'y_true', 'model', 'domain', 'attack', 'decoding', 'repetition_penalty', 'title']
manifest = pd.DataFrame(records)[manifest_columns]
manifest['sample_seed'] = SEED
manifest['dataset_revision'] = sample_metadata['dataset_revision']
manifest.to_csv(ROOT / 'data' / 'sampled_record_ids.csv', index=False)

X = pd.DataFrame([{feature: row[feature] for feature in FEATURES} for row in records], columns=FEATURES, dtype=float)
y = np.asarray([row['y_true'] for row in records], dtype=int)
texts = [row['text'] for row in records]
record_frame = pd.DataFrame(records)
assert len(X) == 10_000 and int((y == 0).sum()) == int((y == 1).sum()) == 5_000
sample_metadata


## 3. InterpretML compatibility gate

Before full evaluation, compare the adapter's probabilities byte-for-byte with the original model on a seeded small sample. Then generate a LIME and Kernel SHAP local explanation using the original model's `predict_proba` through a format-only adapter. No surrogate detector is trained. If a call fails, stop here and report the compatibility blocker.


In [ ]:
gate = compatibility_gate(model, X, n_records=int(CONFIG['explainability']['compatibility_records']), seed=SEED)
assert gate['adapter_exact_probability_parity'] and gate['lime_local_generated'] and gate['shap_kernel_local_generated']
{key: value for key, value in gate.items() if key != 'adapter'}


## 4. Frozen model evaluation

Positive class is AI (`1`); class order comes from `model.classes_` and is expected to be `[0, 1]` = Human, AI. ROC-AUC uses the probability column corresponding to class 1. The confusion matrix uses row/column order Human, AI.


In [ ]:
metrics = evaluate_model(model, X, y)
metrics['excluded_records_during_sample_scan'] = sample_metadata['excluded_counts']
metrics['n_records_excluded_during_sample_scan'] = int(sum(sample_metadata['excluded_counts'].values()))
metrics['eligible_records_seen'] = sample_metadata['eligible_counts_seen']
metrics['sample_seed'] = SEED
metrics['model_card_accuracy_reported_by_author_not_reproduced'] = 0.9603
metrics_path = ROOT / 'outputs' / 'metrics' / 'evaluation_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2, allow_nan=False))

cm = np.asarray(metrics['confusion_matrix_labels_human_ai'])
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(cm, cmap='Blues')
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['Human', 'AI'], yticklabels=['Human', 'AI'], xlabel='Predicted label', ylabel='Ground truth', title='RAID evaluation confusion matrix')
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, f'{value:,}', ha='center', va='center')
fig.colorbar(image, ax=ax, label='Records')
fig.tight_layout()
fig.savefig(ROOT / 'outputs' / 'figures' / 'confusion_matrix.png', dpi=180)
plt.show()
metrics


## 5. Global explanations with InterpretML

Morris sensitivity screens for changes in model output under the method's sampled feature perturbations. Partial dependence marginalizes predictions over the other input columns and assumes feature independence; it may therefore evaluate unrealistic feature combinations. Explanations describe this sampled RAID distribution and do not establish causal effects or universal feature importance.


In [ ]:
# Use a seeded subset for tractable global perturbation explainers.
X_global = X.sample(int(CONFIG['explainability']['global_sample_records']), random_state=SEED).reset_index(drop=True)
explainers = build_explainers(
    model, X_global, seed=SEED,
    shap_background=int(CONFIG['explainability']['shap_background_records']),
    morris_trajectories=int(CONFIG['explainability']['morris_trajectories']),
    morris_num_levels=int(CONFIG['explainability']['morris_num_levels']),
    pdp_num_points=int(CONFIG['explainability']['partial_dependence_points']),
)
morris_exp = explainers['morris'].explain_global(name='Morris sensitivity on RAID sample')
pdp_exp = explainers['pdp'].explain_global(name='Partial dependence on RAID sample')

def jsonable(value):
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)

def explanation_data(exp, keys):
    for key in keys:
        try:
            data = exp.data(key)
            if data is not None:
                return data
        except (KeyError, IndexError, TypeError):
            pass
    return exp.data()

morris_data = explanation_data(morris_exp, [None, 0])
pdp_data = [explanation_data(pdp_exp, [i]) for i in range(len(FEATURES))]
(ROOT / 'outputs' / 'explanations' / 'morris_global.json').write_text(json.dumps(jsonable(morris_data), indent=2))
(ROOT / 'outputs' / 'explanations' / 'partial_dependence_global.json').write_text(json.dumps(jsonable(pdp_data), indent=2))

# InterpretML explanation objects remain available interactively.
from interpret import show
show(morris_exp)
show(pdp_exp, 0)


In [ ]:
# Save an explicit Morris mu* sensitivity figure from the InterpretML explanation data.
names = morris_data.get('names', FEATURES)
scores = np.asarray(morris_data.get('scores', []), dtype=float)
if len(scores) == len(names):
    order = np.argsort(scores)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.barh(np.asarray(names)[order], scores[order], color='#2864a5')
    ax.set_xlabel('Morris mean absolute effect (mu*)')
    ax.set_title('Global Morris sensitivity (InterpretML)')
    fig.tight_layout()
    fig.savefig(ROOT / 'outputs' / 'figures' / 'morris_sensitivity.png', dpi=180)
    plt.show()
else:
    print('Morris chart data unavailable in expected format; serialized explanation retained.')

# Save one PDP plot per feature. Probability scores correspond to model.classes_ order.
class_order = list(model.classes_)
ai_col = class_order.index(1)
fig, axes = plt.subplots(4, 2, figsize=(12, 14))
for i, (feature, data) in enumerate(zip(FEATURES, pdp_data)):
    ax = axes.flat[i]
    grid = np.asarray(data.get('names', data.get('feature_values', [])), dtype=float)
    values = np.asarray(data.get('scores', []), dtype=float)
    if values.ndim == 2:
        values = values[:, ai_col]
    ax.plot(grid, values, marker='o', linewidth=1.5)
    ax.set(xlabel=feature, ylabel='Mean predicted P(AI)', title=feature)
    ax.grid(alpha=.2)
fig.suptitle('Partial dependence of AI probability (InterpretML)')
fig.tight_layout()
fig.savefig(ROOT / 'outputs' / 'figures' / 'partial_dependence.png', dpi=180)
plt.show()


## 6. Local case studies

Select one available record from each requested outcome: correctly classified AI, correctly classified Human, human predicted AI, and AI predicted Human. If an error category does not exist, report it as unavailable. LIME and Kernel SHAP explain the eight numeric inputs only. They do not highlight words or text spans. Both methods approximate black-box behavior and may be unstable, so interpret their directions and magnitudes cautiously.


In [ ]:
pred = np.asarray(model.predict(X), dtype=int)
proba = model.predict_proba(X)
positions = {value: list(model.classes_).index(value) for value in (0, 1)}
cases = {
    'correct_ai': (y == 1) & (pred == 1),
    'correct_human': (y == 0) & (pred == 0),
    'human_misclassified_as_ai': (y == 0) & (pred == 1),
    'ai_misclassified_as_human': (y == 1) & (pred == 0),
}
selected = []
for case_name, mask in cases.items():
    matches = np.flatnonzero(mask)
    if len(matches):
        selected.append((case_name, int(matches[0])))
    else:
        print(f'{case_name}: no example available in evaluation sample')

case_metadata = []
for case_name, idx in selected:
    row = record_frame.iloc[idx]
    case_metadata.append({
        'case': case_name, 'record_index': idx, 'id': row.get('id'), 'ground_truth': int(y[idx]),
        'predicted': int(pred[idx]), 'probabilities': {str(int(c)): float(proba[idx, j]) for j, c in enumerate(model.classes_)},
        'features': {feature: float(X.iloc[idx][feature]) for feature in FEATURES},
        'text_excerpt': str(row['text'])[:500],
    })

case_X = X.iloc[[idx for _, idx in selected]]
case_y = y[[idx for _, idx in selected]]
lime_exp = explainers['lime'].explain_local(case_X, case_y, name='LIME case studies', num_samples=int(CONFIG['explainability']['lime_num_samples']))
shap_exp = explainers['shap'].explain_local(case_X, case_y, name='Kernel SHAP case studies', nsamples=int(CONFIG['explainability']['shap_nsamples']))

# Repeat local approximations with alternate seeds and compare top-3 named inputs.
# If InterpretML's output schema does not expose feature-aligned scores, stability is marked unavailable.
lime_background = X_global.sample(int(CONFIG['explainability']['shap_background_records']), random_state=SEED).astype(float)
lime_alt = __import__('interpret.blackbox', fromlist=['LimeTabular']).LimeTabular(
    explainers['adapter'].predict_proba, lime_background, feature_names=FEATURES, random_state=SEED + 1
).explain_local(case_X, case_y, name='LIME alternate-seed check', num_samples=int(CONFIG['explainability']['lime_num_samples']))
np.random.seed(SEED)
shap_exp_repeat = explainers['shap'].explain_local(case_X, case_y, name='Kernel SHAP repeat check', nsamples=int(CONFIG['explainability']['shap_nsamples']))

def top_features(local_explanation, row_index):
    data = local_explanation.data(row_index)
    names = data.get('names') if isinstance(data, dict) else None
    scores = data.get('scores') if isinstance(data, dict) else None
    if not isinstance(names, (list, tuple)) or scores is None:
        return None
    score_array = np.asarray(scores, dtype=float)
    if score_array.ndim == 2:
        if score_array.shape[-1] == len(names):
            score_array = score_array[positions[1]] if score_array.shape[0] > positions[1] else score_array[-1]
        elif score_array.shape[0] == len(names):
            score_array = score_array[:, positions[1]] if score_array.shape[1] > positions[1] else score_array[:, -1]
    score_array = score_array.reshape(-1)
    if len(score_array) != len(names):
        return None
    matches = []
    for name, score in zip(names, score_array):
        text_name = str(name)
        feature_name = next((feature for feature in FEATURES if feature in text_name), None)
        if feature_name and feature_name not in matches:
            matches.append((feature_name, abs(float(score))))
    return [name for name, _ in sorted(matches, key=lambda pair: pair[1], reverse=True)[:3]] if matches else None

def top3_overlap(left, right):
    if left is None or right is None:
        return None
    a, b = set(left), set(right)
    return len(a & b) / max(1, len(a | b))

local_data = []
for case_pos, (case_name, idx) in enumerate(selected):
    lime_stability = top3_overlap(top_features(lime_exp, case_pos), top_features(lime_alt, case_pos))
    shap_stability = top3_overlap(top_features(shap_exp, case_pos), top_features(shap_exp_repeat, case_pos))
    from interpret import show
    show(lime_exp, case_pos)
    show(shap_exp, case_pos)
    local_data.append({
        'case': case_name, 'metadata': case_metadata[case_pos],
        'lime': jsonable(lime_exp.data(case_pos)), 'kernel_shap': jsonable(shap_exp.data(case_pos)),
        'stability_check': {
            'lime_top3_jaccard': lime_stability,
            'lime_flagged_unstable': lime_stability is not None and lime_stability < 0.5,
            'kernel_shap_top3_jaccard': shap_stability,
            'kernel_shap_flagged_unstable': shap_stability is not None and shap_stability < 0.5,
            'interpretation': 'Scores compare top-three named stylometric inputs across an alternate LIME seed and an independent SHAP sampling run. Unavailable means the explanation schema did not expose a feature-aligned score list.',
        },
        'interpretation': (
            f"LIME ranked {top_features(lime_exp, case_pos)} highest and Kernel SHAP ranked {top_features(shap_exp, case_pos)} highest among the numerical inputs when these scores could be aligned. These are local approximations of the detector's response, not causes, and they do not identify words or text spans."
            if top_features(lime_exp, case_pos) or top_features(shap_exp, case_pos)
            else 'The local explanation reports approximate scores for the eight numerical inputs. The score schema did not support a stable feature ranking; these are not causal effects and do not identify text spans.'
        ),
    })
(ROOT / 'outputs' / 'explanations' / 'local_case_studies.json').write_text(json.dumps(local_data, indent=2))
local_data


## 7. Feature-space what-if analysis

For up to four real sampled records, change one numerical input at a time to observed 10th, 50th, and 90th percentile values, keeping all other values fixed. Report model labels and class probabilities. These feature vectors do not necessarily correspond to valid texts; this is a feature-space sensitivity experiment, not a causal intervention or editing recommendation.


In [ ]:
what_if_records = [records[idx] for _, idx in selected[:4]]
what_if = one_feature_what_if(model, X, what_if_records, quantiles=(0.1, 0.5, 0.9))
what_if.to_csv(ROOT / 'outputs' / 'explanations' / 'what_if_results.csv', index=False)
what_if.head(24)


## 8. Experiment report and limitations

The generated report records measured metrics separately from the model card's author-reported 96.03%. It includes exact data/model revisions, sample counts, exclusions, runtime versions, case availability, and explainer settings. The report is intended for technical review, not authorship adjudication.


In [ ]:
versions = {
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'scikit_learn': sklearn.__version__, 'interpret': __import__('importlib.metadata', fromlist=['version']).version('interpret'),
    'datasets': datasets.__version__, 'huggingface_hub': huggingface_hub.__version__, 'joblib': joblib.__version__,
}
summary = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'model': model_metadata, 'dataset': sample_metadata, 'metrics': metrics, 'versions': versions,
    'compatibility_gate': {key: value for key, value in gate.items() if key != 'adapter'},
    'local_cases_available': [name for name, _ in selected],
    'explainer_settings': {
        'seed': SEED, 'morris_global_records': len(X_global), 'morris_trajectories': 200,
        'morris_num_levels': int(CONFIG['explainability']['morris_num_levels']),
        'partial_dependence_points': 10, 'lime_num_samples': 3000,
        'kernel_shap_background_records': 50, 'kernel_shap_nsamples': 256,
    },
    'limitations': [
        'The model uses eight surface-level stylometric features and may not generalize across domains, generators, attacks, or time.',
        'RAID is an independent source, but source overlap with the detector training corpus cannot be fully ruled out from available identifiers.',
        'LIME and Kernel SHAP are approximations; stability can depend on perturbation and background samples.',
        'Morris sensitivity and partial dependence describe the sampled evaluation distribution; neither establishes causality.',
        'Partial dependence can evaluate implausible combinations when features are dependent.',
        'Feature-space what-if rows need not correspond to valid modified text.',
        'Detector predictions are not proof of authorship and must not be used as individual accusations.',
    ],
}
report_path = ROOT / 'outputs' / 'reports' / 'experiment_report.json'
report_path.write_text(json.dumps(jsonable(summary), indent=2, allow_nan=False))
print('Wrote:', report_path)
final_report = f"""# RAID Detector Evaluation Report

Generated: {summary['generated_at_utc']}

## Frozen model

- Repository/revision: `{model_metadata['repo_id']}` at `{model_metadata['revision']}`
- Artifact: `{model_metadata['filename']}` (SHA-256 `{model_metadata['sha256']}`)
- Class mapping: 0 = Human, 1 = AI (positive class)

## Evaluation

- Sample: {len(y):,} records ({int((y == 0).sum()):,} Human, {int((y == 1).sum()):,} AI), RAID seed {SEED}
- Accuracy: {metrics['accuracy']:.4f}
- Precision / recall / F1 for AI: {metrics['precision_ai']:.4f} / {metrics['recall_ai']:.4f} / {metrics['f1_ai']:.4f}
- ROC-AUC for AI: {metrics['roc_auc_ai']:.4f}
- Confusion matrix (Human, AI): `{metrics['confusion_matrix_labels_human_ai']}`
- Exclusion counts: `{sample_metadata['excluded_counts']}`

The model card's 96.03% accuracy is the author's reported result on a different evaluation and is not reproduced here.

## Interpretability and limits

Morris sensitivity and partial dependence summarize the sampled feature space. LIME and Kernel SHAP approximate local behavior for the four available outcome categories; their contributions are not causal and explain numeric stylometric inputs, not text spans. What-if results modify one numerical input while holding the other seven fixed; changed vectors may not represent valid text. RAID provides an independent source from the model card's stated training set, though exact source overlap cannot be fully ruled out. A detector prediction is not proof of authorship.

See `outputs/metrics/evaluation_metrics.json`, `outputs/explanations/`, and `outputs/figures/` for generated details.
"""
(ROOT / 'outputs' / 'reports' / 'final_report.md').write_text(final_report)

print('Measured accuracy:', metrics['accuracy'])
print('Model card reported accuracy (separate):', metrics['model_card_accuracy_reported_by_author_not_reproduced'])
